# ⚡ SpaceGuard — Mission Energy Monitor
### Sistema Inteligente de Monitoramento de Missão Espacial Experimental

---

**FIAP — Ciências da Computação | Turma 1CCPW | 1º Semestre 2026**  
**Disciplina:** Soluções em Energias Renováveis e Sustentáveis  

| Integrante | RM |
|---|---|
| Henrique Eduardo da Silveira | 571803 |
| Felipe Elze da Silva | 572024 |

---

## Sobre o Projeto

O **SpaceGuard** é um sistema de monitoramento inteligente para missões espaciais experimentais.  
Ele simula e analisa dados de **temperatura**, **energia solar**, **comunicação** e **status dos módulos** operacionais de uma nave em órbita baixa (LEO), gerando **alertas automáticos** e **decisões corretivas** via lógica condicional avançada.

### Conceitos aplicados:
- 🔋 **Energia renovável (fotovoltaica)** — painéis solares como fonte primária
- 🌱 **Sustentabilidade** — eficiência energética e gestão de consumo
- 🤖 **IA introdutória** — motor de decisão baseado em lógica condicional multi-camada
- 📊 **Visualização de dados** — dashboard interativo HTML + gráficos matplotlib

---
## Célula 1 — Instalação de dependências

In [ ]:
# Instalação das bibliotecas necessárias
# (matplotlib e numpy já vêm pré-instalados no Colab)
import subprocess
subprocess.run(['pip', 'install', 'matplotlib', 'numpy', '--quiet'], check=True)
print('✅ Dependências OK')

---
## Célula 2 — Imports e configurações

In [ ]:
import random
import time
import datetime
import math
import json
from dataclasses import dataclass, field, asdict
from typing import List
from enum import Enum
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Estilo dos gráficos
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#030a14',
    'axes.facecolor': '#0a1628',
    'axes.edgecolor': '#0d2440',
    'axes.labelcolor': '#4a6080',
    'text.color': '#c8deff',
    'xtick.color': '#4a6080',
    'ytick.color': '#4a6080',
    'grid.color': '#0d2440',
    'grid.alpha': 0.6,
    'font.family': 'monospace',
})

print('✅ Imports carregados com sucesso')

---
## Célula 3 — Estrutura de dados da missão

In [ ]:
# ──────────────────────────────────────────────────────────────
#  ENUMS
# ──────────────────────────────────────────────────────────────

class AlertLevel(Enum):
    OK      = "OK"
    INFO    = "INFO"
    AVISO   = "AVISO"
    CRITICO = "CRÍTICO"

class ModuleStatus(Enum):
    OPERACIONAL = "OPERACIONAL"
    DEGRADADO   = "DEGRADADO"
    FALHA       = "FALHA"
    STANDBY     = "STANDBY"

# ──────────────────────────────────────────────────────────────
#  DATACLASSES
# ──────────────────────────────────────────────────────────────

@dataclass
class ThermalData:
    """Temperatura dos subsistemas"""
    paineis_solares_c: float = 0.0      # °C — painéis fotovoltaicos
    bateria_c: float = 0.0              # °C — pacote de baterias Li-Ion
    computador_bordo_c: float = 0.0     # °C — processador do computador de bordo

@dataclass
class EnergyData:
    """Sistema energético (fonte: painéis solares fotovoltaicos)"""
    geracao_solar_w: float = 0.0        # Watts gerados pelos painéis
    consumo_total_w: float = 0.0        # Watts consumidos pelos módulos
    bateria_pct: float = 0.0            # % de carga da bateria
    tensao_barramento_v: float = 0.0    # Tensão do barramento elétrico (V)
    eficiencia_pct: float = 0.0         # Eficiência dos painéis fotovoltaicos (%)
    em_eclipse: bool = False            # True = nave na sombra da Terra

@dataclass
class CommunicationData:
    """Sistema de comunicação com estação terrestre"""
    link_ativo: bool = True
    sinal_dbm: float = 0.0             # Potência do sinal (dBm)
    taxa_dados_kbps: float = 0.0       # Taxa de transferência (Kbps)
    latencia_ms: float = 0.0           # Latência round-trip (ms)
    pacotes_perdidos_pct: float = 0.0   # % de pacotes perdidos

@dataclass
class ModuleData:
    """Status dos módulos operacionais da nave"""
    propulsao: ModuleStatus = ModuleStatus.OPERACIONAL
    suporte_vida: ModuleStatus = ModuleStatus.OPERACIONAL
    cientifico: ModuleStatus = ModuleStatus.OPERACIONAL
    navegacao: ModuleStatus = ModuleStatus.OPERACIONAL
    comunicacao: ModuleStatus = ModuleStatus.OPERACIONAL
    energia: ModuleStatus = ModuleStatus.OPERACIONAL

@dataclass
class Alert:
    """Alerta gerado pelo motor de IA"""
    nivel: AlertLevel
    modulo: str
    mensagem: str
    acao_recomendada: str
    timestamp: str = field(default_factory=lambda: datetime.datetime.now().strftime("%H:%M:%S"))

@dataclass
class MissionSnapshot:
    """Snapshot completo da missão em um instante"""
    missao_id: str = "GS-2026-ALPHA"
    timestamp: str = ""
    ciclo: int = 0
    altitude_km: float = 408.0
    velocidade_kms: float = 7.66
    orbita_numero: int = 1
    fase_orbital: str = "ILUMINAÇÃO"
    thermal: ThermalData = field(default_factory=ThermalData)
    energy: EnergyData = field(default_factory=EnergyData)
    communication: CommunicationData = field(default_factory=CommunicationData)
    modules: ModuleData = field(default_factory=ModuleData)
    alerts: List[Alert] = field(default_factory=list)
    score_saude: int = 100

print('✅ Estruturas de dados definidas')

---
## Célula 4 — Simulador orbital

In [ ]:
class OrbitalSimulator:
    """
    Simula dados físicos de uma nave em LEO (Low Earth Orbit ~408 km).
    Modela:
    - Ciclos de iluminação/eclipse (período orbital ~92 minutos)
    - Geração fotovoltaica proporcional à incidência solar
    - Modelo de bateria com carga/descarga e eficiência coulombiana
    - Variações térmicas entre iluminação e eclipse (-100°C a +90°C)
    - Anomalias aleatórias em módulos (probabilidade 8% por ciclo)
    """

    def __init__(self):
        self.ciclo = 0
        self.orbita = 1
        self.fase = 0.0          # graus de fase orbital (0-360)
        self.anomalia_ativa = False
        self.anomalia_modulo = None
        self.historico_bateria = [85.0]

    def _fase_orbital(self):
        """Avança a fase orbital e retorna (em_eclipse, fator_solar)"""
        self.fase = (self.fase + 4.0) % 360.0
        if self.fase < 4.0:
            self.orbita += 1
        em_eclipse = 128.0 < self.fase < 256.0
        fator_solar = 0.0 if em_eclipse else abs(math.sin(math.radians(self.fase)))
        return em_eclipse, fator_solar

    def _gerar_anomalia(self):
        """Injeta/remove anomalias aleatórias nos módulos"""
        if not self.anomalia_ativa and random.random() < 0.08:
            self.anomalia_modulo = random.choice(["propulsao", "cientifico", "comunicacao", "energia"])
            self.anomalia_ativa = True
        elif self.anomalia_ativa and random.random() < 0.3:
            self.anomalia_ativa = False
            self.anomalia_modulo = None

    def generate(self, ciclo: int) -> MissionSnapshot:
        """Gera um snapshot completo da missão"""
        self.ciclo = ciclo
        self._gerar_anomalia()
        em_eclipse, fator_solar = self._fase_orbital()

        snap = MissionSnapshot()
        snap.ciclo = ciclo
        snap.timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        snap.altitude_km = 408.0 + random.uniform(-2.0, 2.0)
        snap.velocidade_kms = 7.66 + random.uniform(-0.01, 0.01)
        snap.orbita_numero = self.orbita
        snap.fase_orbital = "ECLIPSE" if em_eclipse else "ILUMINAÇÃO"

        # Energia
        anomalia_e = 0.4 if (self.anomalia_ativa and self.anomalia_modulo == "energia") else 1.0
        geracao = max(0.0, 8200.0 * fator_solar * anomalia_e + random.gauss(0, 80))
        consumo = max(2000.0, random.gauss(3800, 150))
        bat_ant = self.historico_bateria[-1]
        delta = -(consumo / 100000.0) if em_eclipse else ((geracao - consumo) / 150000.0)
        bat = max(5.0, min(100.0, bat_ant + delta * random.uniform(0.85, 1.1)))
        self.historico_bateria.append(bat)
        if len(self.historico_bateria) > 50: self.historico_bateria.pop(0)
        tensao = 28.0 + (bat - 50.0) * 0.1 + random.gauss(0, 0.3)

        snap.energy = EnergyData(
            geracao_solar_w=round(geracao, 1),
            consumo_total_w=round(consumo, 1),
            bateria_pct=round(bat, 1),
            tensao_barramento_v=round(tensao, 2),
            eficiencia_pct=round(max(0, random.gauss(28.5, 1.2) * anomalia_e), 1),
            em_eclipse=em_eclipse,
        )

        # Temperatura
        temp_bat = 15.0 + (bat - 50.0) * 0.2 + random.gauss(0, 1.5)
        if self.anomalia_ativa and self.anomalia_modulo == "energia": temp_bat += 18.0
        snap.thermal = ThermalData(
            paineis_solares_c=round((-100.0 if em_eclipse else 60.0) + random.gauss(0, 8), 1),
            bateria_c=round(temp_bat, 1),
            computador_bordo_c=round(42.0 + random.gauss(0, 3.0), 1),
        )

        # Comunicação
        link_ok = not (self.anomalia_ativa and self.anomalia_modulo == "comunicacao")
        snap.communication = CommunicationData(
            link_ativo=link_ok,
            sinal_dbm=round(random.gauss(-85, 8) if link_ok else random.gauss(-125, 5), 1),
            taxa_dados_kbps=round(max(0, random.gauss(1200, 120) if link_ok else random.gauss(80, 40)), 0),
            latencia_ms=round(350.0 + random.gauss(0, 20), 0),
            pacotes_perdidos_pct=round(random.uniform(0, 2) if link_ok else random.uniform(15, 45), 1),
        )

        # Módulos
        def s(nome):
            if self.anomalia_ativa and self.anomalia_modulo == nome:
                return random.choice([ModuleStatus.DEGRADADO, ModuleStatus.FALHA])
            return ModuleStatus.OPERACIONAL

        snap.modules = ModuleData(
            propulsao=s("propulsao"),
            suporte_vida=ModuleStatus.OPERACIONAL,
            cientifico=s("cientifico"),
            navegacao=ModuleStatus.OPERACIONAL,
            comunicacao=s("comunicacao") if not link_ok else ModuleStatus.OPERACIONAL,
            energia=s("energia"),
        )

        return snap

print('✅ Simulador orbital definido')

---
## Célula 5 — Motor de IA (tomada de decisão)

In [ ]:
class AIDecisionEngine:
    """
    Motor de inteligência artificial baseado em lógica condicional multi-camada.

    Funcionamento:
    1. Recebe um MissionSnapshot com todos os dados da missão
    2. Analisa cada subsistema contra limites operacionais definidos
    3. Gera alertas categorizados (OK / INFO / AVISO / CRÍTICO)
    4. Calcula o Score de Saúde da Missão (0–100)
    5. Retorna o snapshot enriquecido com alertas e score

    Subsistemas analisados:
    - Energia: bateria, tensão, geração solar
    - Temperatura: painéis, bateria, computador
    - Comunicação: link, sinal, pacotes perdidos
    - Módulos: status de cada módulo operacional
    """

    def analyze(self, snap: MissionSnapshot) -> MissionSnapshot:
        alerts = []
        score = 100

        # ── Energia ──────────────────────────────────────────
        bat = snap.energy.bateria_pct
        if bat < 10:
            alerts.append(Alert(AlertLevel.CRITICO, "BATERIA",
                f"Nível crítico: {bat}% — risco de perda de energia",
                "EMERGÊNCIA: Desligar sistemas não-essenciais. Ativar modo sobrevivência."))
            score -= 35
        elif bat < 20:
            alerts.append(Alert(AlertLevel.AVISO, "BATERIA",
                f"Carga baixa: {bat}%",
                "Reduzir carga científica. Aguardar saída de eclipse."))
            score -= 15
        elif bat > 97:
            alerts.append(Alert(AlertLevel.INFO, "BATERIA",
                f"Bateria no limite superior: {bat}%",
                "Considerar desvio de carga para evitar sobretensão."))
            score -= 3

        tensao = snap.energy.tensao_barramento_v
        if tensao < 24.0 or tensao > 32.0:
            alerts.append(Alert(AlertLevel.CRITICO, "ENERGIA",
                f"Tensão fora de especificação: {tensao}V (nominal: 24–32V)",
                "Acionar regulador de tensão backup. Verificar células fotovoltaicas."))
            score -= 25

        if not snap.energy.em_eclipse and snap.energy.geracao_solar_w < 500:
            alerts.append(Alert(AlertLevel.AVISO, "PAINÉIS SOLARES",
                f"Geração anormalmente baixa em iluminação: {snap.energy.geracao_solar_w}W",
                "Verificar orientação dos painéis. Possível degradação ou obstrução."))
            score -= 20

        # ── Temperatura ───────────────────────────────────────
        tb = snap.thermal.bateria_c
        if tb > 38:
            nivel = AlertLevel.CRITICO if tb >= 44 else AlertLevel.AVISO
            alerts.append(Alert(nivel, "TEMPERATURA",
                f"Temperatura da bateria elevada: {tb}°C (máx nominal: 38°C)",
                "Ativar sistema de resfriamento. Reduzir corrente de carga."))
            score -= 20 if tb >= 44 else 10
        elif tb < -10:
            alerts.append(Alert(AlertLevel.AVISO, "TEMPERATURA",
                f"Bateria abaixo da temperatura mínima: {tb}°C",
                "Ativar aquecedores da bateria. Aguardar normalização térmica."))
            score -= 10

        tc = snap.thermal.computador_bordo_c
        if tc > 60:
            alerts.append(Alert(AlertLevel.CRITICO, "COMPUTADOR DE BORDO",
                f"Processador superaquecido: {tc}°C",
                "Reduzir carga computacional. Ativar dissipador térmico passivo."))
            score -= 20

        # ── Comunicação ───────────────────────────────────────
        if not snap.communication.link_ativo:
            alerts.append(Alert(AlertLevel.CRITICO, "COMUNICAÇÃO",
                "Link com estação terrestre perdido",
                "Executar protocolo de reconexão. Ativar antena de contingência."))
            score -= 30
        else:
            if snap.communication.sinal_dbm < -110:
                alerts.append(Alert(AlertLevel.AVISO, "COMUNICAÇÃO",
                    f"Sinal fraco: {snap.communication.sinal_dbm} dBm",
                    "Reorientar antena. Aumentar potência de transmissão."))
                score -= 10
            if snap.communication.pacotes_perdidos_pct > 10:
                alerts.append(Alert(AlertLevel.AVISO, "COMUNICAÇÃO",
                    f"Alta perda de pacotes: {snap.communication.pacotes_perdidos_pct}%",
                    "Ativar protocolo de retransmissão ARQ."))
                score -= 8

        # ── Módulos ───────────────────────────────────────────
        for nome, status in [
            ("PROPULSÃO", snap.modules.propulsao),
            ("SUPORTE VIDA", snap.modules.suporte_vida),
            ("CIENTÍFICO", snap.modules.cientifico),
            ("NAVEGAÇÃO", snap.modules.navegacao),
            ("COMUNICAÇÃO", snap.modules.comunicacao),
            ("ENERGIA", snap.modules.energia),
        ]:
            if status == ModuleStatus.FALHA:
                alerts.append(Alert(AlertLevel.CRITICO, nome,
                    f"Módulo {nome} em FALHA",
                    f"Isolar módulo. Acionar sistema redundante de backup."))
                score -= 25
            elif status == ModuleStatus.DEGRADADO:
                alerts.append(Alert(AlertLevel.AVISO, nome,
                    f"Módulo {nome} DEGRADADO",
                    f"Monitoramento contínuo. Preparar protocolo de contingência."))
                score -= 12

        if not alerts:
            alerts.append(Alert(AlertLevel.OK, "SISTEMA",
                "Todos os sistemas dentro dos parâmetros nominais",
                "Continuar monitoramento de rotina."))

        snap.alerts = alerts
        snap.score_saude = max(0, min(100, score))
        return snap

print('✅ Motor de IA definido')

---
## Célula 6 — Funções de visualização (matplotlib)

In [ ]:
COR = {
    'verde':   '#00ff88',
    'azul':    '#00d4ff',
    'amarelo': '#ffcc00',
    'vermelho':'#ff4444',
    'roxo':    '#bb88ff',
    'texto':   '#c8deff',
    'mudo':    '#4a6080',
    'bg':      '#030a14',
    'panel':   '#0a1628',
}

def cor_bateria(pct):
    if pct > 40: return COR['verde']
    if pct > 20: return COR['amarelo']
    return COR['vermelho']

def cor_temp(val, warn_max):
    if val > warn_max: return COR['vermelho']
    if val > warn_max * 0.85: return COR['amarelo']
    return COR['verde']

def plotar_dashboard(snap: MissionSnapshot, historico: list):
    """Gera o dashboard completo com 6 painéis matplotlib"""
    fig = plt.figure(figsize=(18, 12), facecolor=COR['bg'])
    fig.suptitle(
        f'⚡ SPACEGUARD — MISSION ENERGY MONITOR | {snap.missao_id} | Ciclo #{snap.ciclo:04d} | {snap.timestamp}',
        fontsize=13, color=COR['azul'], fontweight='bold', y=0.98
    )

    gs = fig.add_gridspec(3, 4, hspace=0.45, wspace=0.35)

    # ── Painel 1: Gauge de Saúde ──────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_facecolor(COR['panel'])
    score = snap.score_saude
    cor_s = COR['verde'] if score >= 70 else (COR['amarelo'] if score >= 40 else COR['vermelho'])
    wedges, _ = ax1.pie(
        [score, 100 - score],
        colors=[cor_s, '#0d2440'],
        startangle=90,
        wedgeprops={'width': 0.45, 'edgecolor': COR['bg'], 'linewidth': 2}
    )
    ax1.text(0, 0, f'{score}', ha='center', va='center', fontsize=28, color=cor_s, fontweight='bold')
    ax1.text(0, -0.35, 'SAÚDE /100', ha='center', fontsize=8, color=COR['mudo'])
    ax1.set_title('ÍNDICE DE SAÚDE', fontsize=9, color=COR['azul'], pad=8)

    # ── Painel 2: Energia atual ───────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_facecolor(COR['panel'])
    e = snap.energy
    categorias = ['Geração\nSolar', 'Consumo\nTotal']
    valores = [e.geracao_solar_w, e.consumo_total_w]
    cores_bar = [COR['verde'] if e.geracao_solar_w >= e.consumo_total_w else COR['amarelo'], COR['vermelho']]
    bars = ax2.bar(categorias, valores, color=cores_bar, width=0.5, edgecolor=COR['bg'], linewidth=1.5)
    for bar, val in zip(bars, valores):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                 f'{val:.0f}W', ha='center', va='bottom', fontsize=9, color=COR['texto'])
    ax2.set_ylim(0, 9500)
    ax2.set_title('BALANÇO ENERGÉTICO', fontsize=9, color=COR['azul'], pad=8)
    ax2.set_ylabel('Watts', fontsize=8, color=COR['mudo'])
    ax2.tick_params(axis='both', labelsize=8)

    # ── Painel 3: Status dos módulos ──────────────────────────
    ax3 = fig.add_subplot(gs[0, 2:4])
    ax3.set_facecolor(COR['panel'])
    ax3.set_xlim(0, 6)
    ax3.set_ylim(0, 2)
    ax3.axis('off')
    ax3.set_title('STATUS DOS MÓDULOS OPERACIONAIS', fontsize=9, color=COR['azul'], pad=8)

    modulos = [
        ('PROPULSÃO',    snap.modules.propulsao),
        ('SUPORTE VIDA', snap.modules.suporte_vida),
        ('CIENTÍFICO',   snap.modules.cientifico),
        ('NAVEGAÇÃO',    snap.modules.navegacao),
        ('COMUNICAÇÃO',  snap.modules.comunicacao),
        ('ENERGIA',      snap.modules.energia),
    ]
    cor_mod = {'OPERACIONAL': COR['verde'], 'DEGRADADO': COR['amarelo'], 'FALHA': COR['vermelho'], 'STANDBY': COR['azul']}
    icone_mod = {'OPERACIONAL': '✓', 'DEGRADADO': '⚠', 'FALHA': '✗', 'STANDBY': '○'}

    for i, (nome, status) in enumerate(modulos):
        x = (i % 3) * 2 + 0.5
        y = 1.4 if i < 3 else 0.4
        cor = cor_mod[status.value]
        icone = icone_mod[status.value]
        circle = plt.Circle((x + 0.15, y + 0.1), 0.25, color=cor, alpha=0.15)
        ax3.add_patch(circle)
        ax3.text(x + 0.15, y + 0.12, icone, ha='center', va='center', fontsize=14, color=cor, fontweight='bold')
        ax3.text(x + 0.55, y + 0.22, nome, ha='left', va='center', fontsize=8, color=COR['texto'])
        ax3.text(x + 0.55, y + 0.02, status.value, ha='left', va='center', fontsize=7, color=cor)

    # ── Painel 4: Histórico Bateria ───────────────────────────
    if len(historico) > 1:
        ax4 = fig.add_subplot(gs[1, 0:2])
        ax4.set_facecolor(COR['panel'])
        ts = list(range(len(historico[-30:])))
        bat_h = [h.energy.bateria_pct for h in historico[-30:]]
        ax4.plot(ts, bat_h, color=COR['verde'], linewidth=2)
        ax4.fill_between(ts, bat_h, alpha=0.15, color=COR['verde'])
        ax4.axhline(20, color=COR['vermelho'], linestyle='--', linewidth=1, alpha=0.7, label='Mín. crítico')
        ax4.axhline(40, color=COR['amarelo'], linestyle='--', linewidth=1, alpha=0.5, label='Mín. aviso')
        ax4.set_ylim(0, 105)
        ax4.set_title('HISTÓRICO — BATERIA (%)', fontsize=9, color=COR['azul'], pad=8)
        ax4.set_ylabel('%', fontsize=8, color=COR['mudo'])
        ax4.legend(fontsize=7, loc='upper right')
        ax4.tick_params(axis='both', labelsize=7)
        # Shade eclipse periods
        for i, h in enumerate(historico[-30:]):
            if h.energy.em_eclipse:
                ax4.axvspan(i - 0.5, i + 0.5, alpha=0.08, color=COR['roxo'])

    # ── Painel 5: Histórico Energia Solar ────────────────────
    if len(historico) > 1:
        ax5 = fig.add_subplot(gs[1, 2:4])
        ax5.set_facecolor(COR['panel'])
        ts = list(range(len(historico[-30:])))
        solar_h = [h.energy.geracao_solar_w for h in historico[-30:]]
        cons_h  = [h.energy.consumo_total_w for h in historico[-30:]]
        ax5.plot(ts, solar_h, color=COR['azul'], linewidth=2, label='Geração Solar')
        ax5.plot(ts, cons_h,  color=COR['amarelo'], linewidth=1.5, linestyle='--', label='Consumo')
        ax5.fill_between(ts, solar_h, cons_h,
                          where=[s >= c for s, c in zip(solar_h, cons_h)],
                          alpha=0.12, color=COR['verde'], label='Superávit')
        ax5.fill_between(ts, solar_h, cons_h,
                          where=[s < c for s, c in zip(solar_h, cons_h)],
                          alpha=0.12, color=COR['vermelho'], label='Déficit')
        ax5.set_title('HISTÓRICO — ENERGIA SOLAR vs CONSUMO (W)', fontsize=9, color=COR['azul'], pad=8)
        ax5.set_ylabel('Watts', fontsize=8, color=COR['mudo'])
        ax5.legend(fontsize=7, loc='upper right')
        ax5.tick_params(axis='both', labelsize=7)

    # ── Painel 6: Temperatura histórico + alertas ─────────────
    ax6 = fig.add_subplot(gs[2, 0:2])
    ax6.set_facecolor(COR['panel'])
    if len(historico) > 1:
        ts = list(range(len(historico[-30:])))
        tb_h = [h.thermal.bateria_c for h in historico[-30:]]
        tc_h = [h.thermal.computador_bordo_c for h in historico[-30:]]
        ax6.plot(ts, tb_h, color=COR['roxo'], linewidth=2, label='Bateria (°C)')
        ax6.plot(ts, tc_h, color=COR['azul'],  linewidth=1.5, linestyle='--', label='Computador (°C)')
        ax6.axhline(38, color=COR['vermelho'], linestyle=':', linewidth=1, alpha=0.7)
        ax6.set_title('HISTÓRICO — TEMPERATURA', fontsize=9, color=COR['azul'], pad=8)
        ax6.set_ylabel('°C', fontsize=8, color=COR['mudo'])
        ax6.legend(fontsize=7)
        ax6.tick_params(axis='both', labelsize=7)

    # ── Painel 7: Alertas ────────────────────────────────────
    ax7 = fig.add_subplot(gs[2, 2:4])
    ax7.set_facecolor(COR['panel'])
    ax7.axis('off')
    ax7.set_title('ALERTAS DO SISTEMA DE IA', fontsize=9, color=COR['azul'], pad=8)

    cor_nivel = {'OK': COR['verde'], 'INFO': COR['azul'], 'AVISO': COR['amarelo'], 'CRÍTICO': COR['vermelho']}
    y_pos = 0.92
    for alerta in snap.alerts[:5]:
        cor = cor_nivel.get(alerta.nivel.value, COR['texto'])
        ax7.text(0.02, y_pos, f'[{alerta.nivel.value:8s}]', transform=ax7.transAxes,
                 fontsize=8, color=cor, fontweight='bold', verticalalignment='top')
        ax7.text(0.30, y_pos, f'{alerta.modulo}: {alerta.mensagem}',
                 transform=ax7.transAxes, fontsize=7.5, color=COR['texto'],
                 verticalalignment='top', wrap=True)
        ax7.text(0.30, y_pos - 0.09, f'▶ {alerta.acao_recomendada}',
                 transform=ax7.transAxes, fontsize=6.5, color=COR['mudo'],
                 verticalalignment='top', style='italic')
        y_pos -= 0.20

    plt.savefig('/content/spaceguard_dashboard.png', dpi=130, bbox_inches='tight', facecolor=COR['bg'])
    plt.show()
    print(f'✅ Dashboard salvo: /content/spaceguard_dashboard.png')

print('✅ Funções de visualização definidas')

---
## Célula 7 — Executar o monitor (N ciclos)

In [ ]:
# ─────────────────────────────────────────────
#  CONFIGURAÇÃO — altere aqui conforme desejar
# ─────────────────────────────────────────────
N_CICLOS = 30          # Número de ciclos de simulação
INTERVALO_S = 0        # Delay entre ciclos em segundos (0 = rápido para demo)
PLOTAR_A_CADA = 10     # Gerar gráfico a cada N ciclos
# ─────────────────────────────────────────────

simulador = OrbitalSimulator()
motor_ia  = AIDecisionEngine()
historico = []

print('🚀 Iniciando SpaceGuard Mission Monitor...')
print(f'   Missão: GS-2026-ALPHA | {N_CICLOS} ciclos de simulação')
print('─' * 60)

for ciclo in range(1, N_CICLOS + 1):
    raw = simulador.generate(ciclo)
    snap = motor_ia.analyze(raw)
    historico.append(snap)

    # Resumo textual por ciclo
    e = snap.energy
    fase_icon = '🌑' if snap.fase_orbital == 'ECLIPSE' else '☀️'
    score_icon = '🟢' if snap.score_saude >= 70 else ('🟡' if snap.score_saude >= 40 else '🔴')
    alertas_crit = sum(1 for a in snap.alerts if a.nivel == AlertLevel.CRITICO)
    alertas_avis = sum(1 for a in snap.alerts if a.nivel == AlertLevel.AVISO)

    print(f'Ciclo {ciclo:03d} {fase_icon} | Bat: {e.bateria_pct:5.1f}% | '
          f'Solar: {e.geracao_solar_w:6.0f}W | Consumo: {e.consumo_total_w:5.0f}W | '
          f'Saúde: {score_icon}{snap.score_saude:3d}/100 | '
          f'Alertas: 🔴{alertas_crit} 🟡{alertas_avis}')

    if ciclo % PLOTAR_A_CADA == 0 or ciclo == N_CICLOS:
        print(f'\n  📊 Gerando dashboard — Ciclo {ciclo}...')
        plotar_dashboard(snap, historico)
        print()

    if INTERVALO_S > 0:
        time.sleep(INTERVALO_S)

print('─' * 60)
print(f'✅ Simulação concluída: {N_CICLOS} ciclos | Score final: {snap.score_saude}/100')

---
## Célula 8 — Relatório final de estatísticas

In [ ]:
# Estatísticas consolidadas da simulação
print('\n' + '='*60)
print('  📋 RELATÓRIO FINAL — SpaceGuard Mission GS-2026-ALPHA')
print('='*60)

bat_media = sum(h.energy.bateria_pct for h in historico) / len(historico)
solar_media = sum(h.energy.geracao_solar_w for h in historico) / len(historico)
score_medio = sum(h.score_saude for h in historico) / len(historico)
eclipses = sum(1 for h in historico if h.energy.em_eclipse)
total_criticos = sum(1 for h in historico for a in h.alerts if a.nivel == AlertLevel.CRITICO)
total_avisos   = sum(1 for h in historico for a in h.alerts if a.nivel == AlertLevel.AVISO)
bat_min = min(h.energy.bateria_pct for h in historico)
bat_max = max(h.energy.bateria_pct for h in historico)

print(f'  Ciclos simulados     : {len(historico)}')
print(f'  Ciclos em eclipse    : {eclipses} ({eclipses/len(historico)*100:.1f}%)')
print(f'  Score de saúde médio : {score_medio:.1f}/100')
print(f'  Bateria média        : {bat_media:.1f}% (min: {bat_min:.1f}% / max: {bat_max:.1f}%)')
print(f'  Geração solar média  : {solar_media:.0f} W')
print(f'  Alertas CRÍTICOS     : {total_criticos}')
print(f'  Alertas AVISO        : {total_avisos}')
print('='*60)
print(f'  Henrique Eduardo da Silveira — RM 571803')
print(f'  Felipe Elze da Silva         — RM 572024')
print(f'  FIAP 1CCPW | Soluções em Energias Renováveis | 2026')
print('='*60)